In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open('names.txt', 'r').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [4]:
len(words)

32033

In [7]:
# build a vocabulary of teh characters and mappinmgs to/from intergers

chars = sorted(list(set(''.join(words))))

stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0

itos = {i: s for s, i in stoi.items()}
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [12]:
# buid teh datset

block_size = 3    # context length: how many characters do we take to predict the next one? Here we have take 2 to follow teh paper

X, Y = [], []

for w in words[:5]:

  print('\n'+w)
  context = [0] * block_size

  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', ch)
    context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .

olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .

ava
... ---> a
..a ---> v
.av ---> a
ava ---> .

isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .

sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [16]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [17]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [18]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [19]:
# Lets create a look up embedding

C = torch.randn((27, 2))
C

tensor([[ 0.7560,  0.2345],
        [-0.2164,  1.3144],
        [-0.2602, -2.4986],
        [-0.4111, -0.4369],
        [-2.1861, -1.3817],
        [-1.3685,  1.0333],
        [-0.0323, -0.0675],
        [ 1.4054,  0.5742],
        [ 0.6192, -0.3980],
        [ 0.8500, -0.3328],
        [ 0.0781,  0.6107],
        [ 0.6943,  0.7684],
        [ 0.2585,  1.2894],
        [ 2.0817, -1.5526],
        [ 0.1226,  0.2596],
        [ 0.2167,  0.7142],
        [-1.0105, -2.0880],
        [ 0.3307,  0.1174],
        [ 1.2272, -0.0782],
        [-0.2271, -0.2966],
        [ 0.0747,  0.7502],
        [ 0.1185, -1.5185],
        [ 0.3390,  0.4368],
        [ 0.4932, -1.3110],
        [ 1.3488, -2.1008],
        [ 1.1907,  0.1918],
        [-0.6416,  0.9366]])

In [21]:
C[5]

tensor([-1.3685,  1.0333])

In [20]:
C.dtype

torch.float32

In [23]:
# Lets try to embedded int 5

# we have 27 possible char

# one hot encoding
# F.one_hot(5, num_classes=27)            # ERROR: (position 1) must be Tensor, not int------------------

F.one_hot(torch.tensor(5), num_classes=27)

tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])

In [24]:
F.one_hot(torch.tensor(5), num_classes=27).dtype

torch.int64

In [28]:
# F.one_hot(torch.tensor(5), num_classes=27) @ C          #RuntimeError: expected m1 and m2 to have the same dtype, but got: long int != float

#  our one hot encodeing tensor have dtype as long int int64 and C have dtype of float32
# Pytorch doesn't know how to multiply int into float

# According to me here float makes sense

F.one_hot(torch.tensor(5), num_classes=27).float() @ C

# 1, 27 X 27, 2 = 1, 2

tensor([-1.3685,  1.0333])

In [29]:
C[5]

tensor([-1.3685,  1.0333])

In [32]:
# The results are same
# Thus we will not do one hot encoding and then multiply with C
# we will use C directly as results are the same

In [33]:
C = torch.randn(27, 2)

In [34]:
C[5]

tensor([-1.2860, -0.1074])

In [35]:
C[[5, 6, 7]]

tensor([[-1.2860, -0.1074],
        [-1.7838, -0.4144],
        [ 0.3810,  0.1002]])

In [38]:
C[torch.tensor([5, 6, 7, 7])]

tensor([[-1.2860, -0.1074],
        [-1.7838, -0.4144],
        [ 0.3810,  0.1002],
        [ 0.3810,  0.1002]])

In [39]:
X.shape, Y.shape

(torch.Size([32, 3]), torch.Size([32]))

In [42]:
X[13, 2]

tensor(1)

In [47]:
C[1]

tensor([2.0064, 1.0412])

In [46]:
C[X[13, 2]]

tensor([2.0064, 1.0412])

In [59]:
C[X][13, 2]

tensor([2.0064, 1.0412])

In [55]:
X[:4]

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13]])

In [56]:
C[X][:4]

tensor([[[-0.1947,  1.2643],
         [-0.1947,  1.2643],
         [-0.1947,  1.2643]],

        [[-0.1947,  1.2643],
         [-0.1947,  1.2643],
         [-1.2860, -0.1074]],

        [[-0.1947,  1.2643],
         [-1.2860, -0.1074],
         [ 0.0773, -2.0502]],

        [[-1.2860, -0.1074],
         [ 0.0773, -2.0502],
         [ 0.0773, -2.0502]]])

In [57]:
C[X][:4].shape

torch.Size([4, 3, 2])

In [58]:
X[:4].shape

torch.Size([4, 3])

In [60]:
# embedding
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [61]:
# row = 32
# column = 3
# dim of each cell/value = 2

In [62]:
# lets create hidden layer of our neural network

W1 = torch.randn(6, 100)          # 3x2=6 input and 100 nureons
b1 = torch.randn(100)

In [66]:
# now if we try to do things noramally
# emb @ W1 + b1             # RuntimeError: mat1 and mat2 shapes cannot be multiplied (96x2 and 6x100)------------

In [68]:
emb[:, 0, :]          # 1st char embeddings

tensor([[-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [-1.2860, -0.1074],
        [ 0.0773, -2.0502],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [ 1.2552,  0.3919],
        [-1.1989, -0.4386],
        [-2.5765, -1.1345],
        [-0.1565,  0.2247],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [ 2.0064,  1.0412],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [-2.5765, -1.1345],
        [ 0.5949,  0.8130],
        [ 2.0064,  1.0412],
        [ 0.0290,  0.8419],
        [-1.2860, -0.1074],
        [-1.1989, -0.4386],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [-0.1947,  1.2643],
        [ 0.5949,  0.8130],
        [ 1.2552,  0.3919],
        [ 2.1753, -0.4354],
        [ 1.2803, -0.0616]])

In [70]:
emb[:, 0, :].shape

torch.Size([32, 2])

In [75]:
# we want for all 3 char

# emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]
# 32,2, 32,2, 32,2
# then we will concatinate them along column
# after concat along column -> 32, 3x2 = 32, 6

In [76]:
emb[:, 0, :][:3], emb[:, 1, :][:3], emb[:, 2, :][:3]

(tensor([[-0.1947,  1.2643],
         [-0.1947,  1.2643],
         [-0.1947,  1.2643]]),
 tensor([[-0.1947,  1.2643],
         [-0.1947,  1.2643],
         [-1.2860, -0.1074]]),
 tensor([[-0.1947,  1.2643],
         [-1.2860, -0.1074],
         [ 0.0773, -2.0502]]))

In [77]:
torch.cat(tensors = [emb[:, 0, :][:3], emb[:, 1, :][:3], emb[:, 2, :][:3]], dim = 1)

tensor([[-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643, -1.2860, -0.1074],
        [-0.1947,  1.2643, -1.2860, -0.1074,  0.0773, -2.0502]])

In [87]:
# you can see the matix/tensor they got concat

# refer = https://docs.pytorch.org/docs/2.14/generated/torch.cat.html.  **********************

In [84]:
print(emb[:, 0, :][:3].shape)
print(emb[:, 1, :][:3].shape)
print(emb[:, 2, :][:3].shape, '\n------------------')

torch.cat(tensors = [emb[:, 0, :][:3], emb[:, 1, :][:3], emb[:, 2, :][:3]], dim = 1).shape

torch.Size([3, 2])
torch.Size([3, 2])
torch.Size([3, 2]) 
------------------


torch.Size([3, 6])

In [86]:
# Now lets do for all rows not just [:3]
# we also have to generalize here we are considering 3 char (block_size) but what if we have diif (more or less)

In [88]:
# refer = https://docs.pytorch.org/docs/2.14/generated/torch.unbind.html ********************

# torch.unbind(torch.tensor([[1, 2, 3],
#                            [4, 5, 6],
#                            [7, 8, 9]]))
# (tensor([1, 2, 3]), tensor([4, 5, 6]), tensor([7, 8, 9]))

In [95]:
torch.unbind(emb[:3], 1)

(tensor([[-0.1947,  1.2643],
         [-0.1947,  1.2643],
         [-0.1947,  1.2643]]),
 tensor([[-0.1947,  1.2643],
         [-0.1947,  1.2643],
         [-1.2860, -0.1074]]),
 tensor([[-0.1947,  1.2643],
         [-1.2860, -0.1074],
         [ 0.0773, -2.0502]]))

In [90]:
emb[:, 0, :][:3], emb[:, 1, :][:3], emb[:, 2, :][:3]

(tensor([[-0.1947,  1.2643],
         [-0.1947,  1.2643],
         [-0.1947,  1.2643]]),
 tensor([[-0.1947,  1.2643],
         [-0.1947,  1.2643],
         [-1.2860, -0.1074]]),
 tensor([[-0.1947,  1.2643],
         [-1.2860, -0.1074],
         [ 0.0773, -2.0502]]))

In [91]:
# torch.unbind(emb[:3]) = emb[:, 0, :][:3], emb[:, 1, :][:3], emb[:, 2, :][:3]

In [96]:
torch.cat(tensors = torch.unbind(emb[:3], 1), dim = 1)

tensor([[-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643, -1.2860, -0.1074],
        [-0.1947,  1.2643, -1.2860, -0.1074,  0.0773, -2.0502]])

In [94]:
torch.cat(tensors = [emb[:, 0, :][:3], emb[:, 1, :][:3], emb[:, 2, :][:3]], dim = 1)

tensor([[-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643, -1.2860, -0.1074],
        [-0.1947,  1.2643, -1.2860, -0.1074,  0.0773, -2.0502]])

In [98]:
# now block_size doesn't matter

In [101]:
torch.cat(tensors = torch.unbind(emb, 1), dim = 1)

tensor([[-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643, -1.2860, -0.1074],
        [-0.1947,  1.2643, -1.2860, -0.1074,  0.0773, -2.0502],
        [-1.2860, -0.1074,  0.0773, -2.0502,  0.0773, -2.0502],
        [ 0.0773, -2.0502,  0.0773, -2.0502,  2.0064,  1.0412],
        [-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643,  1.2552,  0.3919],
        [-0.1947,  1.2643,  1.2552,  0.3919, -1.1989, -0.4386],
        [ 1.2552,  0.3919, -1.1989, -0.4386, -2.5765, -1.1345],
        [-1.1989, -0.4386, -2.5765, -1.1345, -0.1565,  0.2247],
        [-2.5765, -1.1345, -0.1565,  0.2247, -2.5765, -1.1345],
        [-0.1565,  0.2247, -2.5765, -1.1345,  2.0064,  1.0412],
        [-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643,  2.0064,  1.0412],
        [-0.1947,  1.2643,  2.0064,  1.0412, -0.1565,  0.2247],
        [ 2.0064,  1.0412, -0.1565,  0.2

In [103]:
torch.cat(tensors = torch.unbind(emb, 1), dim = 1).shape

torch.Size([32, 6])

In [112]:
# there are other methods also to do it
# which can be more EXTREMLY efficient

In [107]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [108]:
a.shape

torch.Size([18])

In [109]:
a.view(3, 3, 2)

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [110]:
a.view(9, 2)

tensor([[ 0,  1],
        [ 2,  3],
        [ 4,  5],
        [ 6,  7],
        [ 8,  9],
        [10, 11],
        [12, 13],
        [14, 15],
        [16, 17]])

In [111]:
a.storage()

/tmp/ipykernel_4118/214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [114]:

# each tensore have underline storage and the storage is just the numbers always as a 1D vector
# and this is how this tensor is represneted in teh computer memory
# Its always the 1D vector.
# But when we call that view, we are manipulating some of the attributes of that tensor that
# dictate how this 1D sequence is interpreted to be am N-dimensional tensor.
# And so, whats happening ther is that NO MEMORY IS BEING CHANGEDM COPIED, MOVED OR CREATED when
# we call .view()
# The storage is identical but when we call .view() some of teh internal attributes of the view of this tensor are being
# manipulated and change
# particular, there is something called storage offset, strides, and shapes, and those
# are manipulated so that this 1D sequence of bytes are seen as differnet N-dimensional arrays.


In [123]:
# Concatination
# this is very inefficienct as it crate whole new tensore with teh whole new storage.
# so memory is being created

In [115]:
emb.shape

torch.Size([32, 3, 2])

In [116]:
emb.view(32, 6)

tensor([[-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643, -1.2860, -0.1074],
        [-0.1947,  1.2643, -1.2860, -0.1074,  0.0773, -2.0502],
        [-1.2860, -0.1074,  0.0773, -2.0502,  0.0773, -2.0502],
        [ 0.0773, -2.0502,  0.0773, -2.0502,  2.0064,  1.0412],
        [-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643,  1.2552,  0.3919],
        [-0.1947,  1.2643,  1.2552,  0.3919, -1.1989, -0.4386],
        [ 1.2552,  0.3919, -1.1989, -0.4386, -2.5765, -1.1345],
        [-1.1989, -0.4386, -2.5765, -1.1345, -0.1565,  0.2247],
        [-2.5765, -1.1345, -0.1565,  0.2247, -2.5765, -1.1345],
        [-0.1565,  0.2247, -2.5765, -1.1345,  2.0064,  1.0412],
        [-0.1947,  1.2643, -0.1947,  1.2643, -0.1947,  1.2643],
        [-0.1947,  1.2643, -0.1947,  1.2643,  2.0064,  1.0412],
        [-0.1947,  1.2643,  2.0064,  1.0412, -0.1565,  0.2247],
        [ 2.0064,  1.0412, -0.1565,  0.2

In [120]:
emb.view(32, 6) == torch.cat(tensors = torch.unbind(emb, 1), dim = 1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [118]:
# Both Concatinate and View produces teh same results
# But the .view() is extremly efficient

In [121]:
# Lets go to hidden layer now and try matix multiplication

In [122]:
# emb.view(32, 6) @ W1 +
# inplace of 32 we can write '-1' and pytorch will automatically calculate it

emb.view(-1, 6) @ W1 + b1

tensor([[-1.3019, -3.6193,  0.2339,  ...,  3.2497,  1.6903, -0.1505],
        [-2.3279, -1.4246,  1.8876,  ..., -0.1233,  1.9422,  0.6890],
        [ 4.1876, -0.9283, -0.0966,  ..., -6.4255,  1.6250,  1.6059],
        ...,
        [-0.2178,  0.9786, -0.8315,  ...,  4.7612,  2.6029,  1.4810],
        [-9.5651,  4.1743,  4.5792,  ..., -1.1221,  1.0232,  1.4301],
        [ 3.1796, -5.6927, -3.2288,  ..., -2.5219, -1.5417, -1.2346]])

In [124]:
# adding activation function 'tanh'

h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
h

tensor([[-0.8622, -0.9986,  0.2297,  ...,  0.9970,  0.9342, -0.1494],
        [-0.9812, -0.8906,  0.9552,  ..., -0.1227,  0.9597,  0.5973],
        [ 0.9995, -0.7298, -0.0963,  ..., -1.0000,  0.9253,  0.9226],
        ...,
        [-0.2145,  0.7525, -0.6813,  ...,  0.9999,  0.9891,  0.9016],
        [-1.0000,  0.9995,  0.9998,  ..., -0.8083,  0.7712,  0.8917],
        [ 0.9965, -1.0000, -0.9969,  ..., -0.9872, -0.9124, -0.8439]])

In [125]:
h.shape

torch.Size([32, 100])

In [126]:
# brodcasting for b1

# emb.view(-1, 6) @ W1
# sahpe is 32, 100

# 32, 100         emb.view(-1, 6) @ W1
#     100         b1

# 32, 100
#  1, 100

# 1, 100 --- row vector
# thus it will be stretch row wise
# 32, 100
# will have identical row

# this is correct behavious we want for our case

In [127]:
# output layer

W2 = torch.randn(100, 27)             # 27 possible char
b2 = torch.randn(27)

In [142]:
torch.set_printoptions(precision=4, sci_mode=False)         # will display all tensor in round(4)

In [143]:
logits = h @ W2 + b2
logits

tensor([[ -8.4795,  -2.1579,   0.6600,   7.9168,  -6.8725,   6.6633,  11.2733,
          -2.0669,   0.6028,   5.5172,   8.3491,  -2.9312, -12.5400,   6.8553,
          11.3104,   9.9497,  -8.0044,  -1.1993,  13.0948,   2.3982,  15.8150,
         -12.7817, -12.3602,   3.0155,  -1.1473,  17.2202, -13.5226],
        [ -7.9714,  -8.0367,  11.2429,  16.0566,  -9.9800,   6.5921,  15.9153,
           0.5816,  -4.0040,  13.5827,  -1.8703,   0.1968,  -9.3326,  -0.5310,
           6.2154,  12.1300,  -2.0438,  -3.8848,  12.2711,   6.2282,  21.7717,
          -7.6515, -18.8956,   4.7042,  -0.4991,  17.4422, -11.4224],
        [ -0.0820, -13.2119,   4.3544,  12.9688,  -4.9215,  21.1423,   4.7213,
          -4.0761,  -3.5995,   7.1688,  -1.2023,  -9.1754,   8.0197, -11.7878,
          -0.6049,   1.3311,  -0.6627,   9.3709,   8.4720,  16.1838,   1.2171,
          13.5006,  -8.8503,   2.6224,   7.1939,  16.4760,   5.2732],
        [ -1.4264, -14.0140,  11.1842,   1.6678,  -5.3510,   6.4813,  12.2287,


In [144]:
logits.shape

torch.Size([32, 27])

In [145]:
counts = logits.exp()

In [146]:
prob = counts / counts.sum(1, keepdim=True)

In [147]:
prob.shape

torch.Size([32, 27])

In [148]:
prob[0].sum()

tensor(1.0000)

In [149]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [150]:
prob[torch.arange(32), Y]             # prob of teh expected output

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0005, 0.0000, 0.0000, 0.0000,
        0.0103, 0.0000, 0.0218, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0135, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.9450])

In [152]:
# loss = mean of negative log likelihood
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(19.2732)

In [153]:
# our goal is to minimize the loss

In [154]:
# Lets put everything together

In [169]:
torch.set_printoptions(precision=4, sci_mode=False)         # will display all tensor in round(4)

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)

W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)

W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]

In [157]:
# Total number of parameters
sum(e.nelement() for e in parameters)

3481

In [163]:
emb = C[X]                                      # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)       # (32, 100)
logits = h @ W2 + b2                            # (32, 27)
# counts = logits.exp()
# prob = counts / counts.sum(1, keepdim=True)
# loss = -prob[torch.arange(32), Y].log().mean()        # tensor(17.7697)
loss = F.cross_entropy(logits, Y)
loss

tensor(17.7697)

In [164]:
# using F.cross_entropy is more efficient then calculating counts, probs
# beacuse each one of this will create a tensore thus occupy

# F.cross_entropy
# pytorch will cluster up all this operation and very offtent we will have th fussed kernels
# that very efficiently evaluate this operations
# ALSO, it help during BACK PROPOGRATION - more efficient because
# analatically, mathematically it will be very simple backward pass to be implemented
# ALSO, internally it is NUMERICALLY very well behaved
# if logits have very high positive number (outside of our dtype ex.: float32) we get count as infinity
# but while using. cross_entropy pytorch internally handles it but subtracting highest num of logit from logit
# this will resul the same outcome (loss) but numerically well behaved

In [165]:
# forward pass
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

tensor(17.7697)

In [171]:
for p in parameters:
  p.requires_grad = True

In [172]:
for _ in range(1000):

  # forward pass
  emb = C[X]    # (32, 3, 2)  - 32 example
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Y)

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  for p in parameters:
    p.data += -0.1 * p.grad

print(loss)

tensor(0.2562, grad_fn=<NllLossBackward0>)


In [173]:
# Lets build for whole dateset now

In [176]:
# buid the datset

block_size = 3    # context length: how many characters do we take to predict the next one? Here we have take 2 to follow teh paper

X, Y = [], []

for w in words:

  context = [0] * block_size

  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [177]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [187]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)

W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)

W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]\

for p in parameters:
  p.requires_grad = True

In [188]:
for _ in range(10):

  # forward pass
  emb = C[X]
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Y)
  print(loss)

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  for p in parameters:
    p.data += -0.1 * p.grad

print(f'loss = {loss:.4f}')

tensor(19.5052, grad_fn=<NllLossBackward0>)
tensor(17.0845, grad_fn=<NllLossBackward0>)
tensor(15.7765, grad_fn=<NllLossBackward0>)
tensor(14.8333, grad_fn=<NllLossBackward0>)
tensor(14.0026, grad_fn=<NllLossBackward0>)
tensor(13.2533, grad_fn=<NllLossBackward0>)
tensor(12.5799, grad_fn=<NllLossBackward0>)
tensor(11.9831, grad_fn=<NllLossBackward0>)
tensor(11.4705, grad_fn=<NllLossBackward0>)
tensor(11.0519, grad_fn=<NllLossBackward0>)
loss = 11.0519


In [189]:
# here the process is very slow

# generally we have to use minibatch

torch.randint(0, X.shape[0], (32,))

tensor([193838,  71076, 186170,  23777, 130805, 181295,  52653, 204678, 206756,
         96624, 175373, 102958, 118659, 145260,  21509,  75576,  96296, 219217,
        209903,  80446,  59125, 113273, 145385, 218879,  60018, 119883, 164375,
        181956, 113955, 181268,  57260,  14434])

In [191]:

# since we are using minibatches
# the quality of our gradient is lower
# So, the direction (gradient direction during back pass) is not as reliable.
# Its not the actual gradient direction
# But the gradient direction is good enough even when
# its estimating only 32 example (we provided 32 size while creating minibatch)

# Its much better to hav teh approx gradient and just make more steps than
# it is to evaluate the excat gradient and take fewer setps


In [195]:
for _ in range(100):

  # minibatch
  ix = torch.randint(0, X.shape[0], (32,))

  # forward pass
  emb = C[X[ix]]
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Y[ix])
  # print(loss)                 # ------- uncomment this to see how fast we get losses

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  for p in parameters:
    p.data += -0.1 * p.grad

print(f'loss = {loss:.4f}')

loss = 2.8663


In [196]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

tensor(2.8743, grad_fn=<NllLossBackward0>)